## tl;dr

Для базового пилота на 5 месяцев требуется **7,3 FTE / 36,5 человеко-месяца**. Fully-loaded ФОТ составляет **14 750 450 ₽** в базовом сценарии и **11 926 200–18 153 850 ₽** по диапазону ставок. После резерва 15% базовый ориентир финансирования ФОТ — **16 963 018 ₽**.


## Context & Methods

Режим: расчётный companion к executive-отчёту. Аудитория: владелец продукта, Finance/HR и заказчик пилота. Валюта — рубль, ставки — начисленная месячная зарплата до НДФЛ.

### Key Assumptions

- горизонт базового проекта: 5 месяцев; чувствительность: 4–6 месяцев;
- страховые взносы: 30% ко всему ФОТ как консервативный плановый тариф;
- организационные накладные: 10% после страховых взносов;
- ML/LLM, промышленная интеграция и независимый аудит считаются отдельно;
- источники и ссылки сохранены в `evidence.md`.


In [1]:
months = 5
insurance_multiplier = 1.30
overhead_multiplier = 1.10
loaded_multiplier = insurance_multiplier * overhead_multiplier
contingency = 0.15
planning_fx_rub_per_usd = 85

roles = [
    {"role": "Backend lead / архитектор", "fte": 1.0, "low": 420_000, "base": 497_000, "high": 550_000},
    {"role": "Backend Python инженер", "fte": 1.0, "low": 220_000, "base": 278_000, "high": 350_000},
    {"role": "Frontend React инженер", "fte": 1.0, "low": 210_000, "base": 260_000, "high": 320_000},
    {"role": "QA automation", "fte": 1.0, "low": 180_000, "base": 230_000, "high": 300_000},
    {"role": "Инженер-технолог ЭЛОУ-АВТ", "fte": 1.0, "low": 200_000, "base": 250_000, "high": 320_000},
    {"role": "Product / Project manager", "fte": 0.5, "low": 200_000, "base": 260_000, "high": 330_000},
    {"role": "Методист / human factors", "fte": 0.5, "low": 160_000, "base": 180_000, "high": 220_000},
    {"role": "DevOps / SRE", "fte": 0.5, "low": 220_000, "base": 280_000, "high": 350_000},
    {"role": "AppSec / ИБ", "fte": 0.5, "low": 200_000, "base": 250_000, "high": 330_000},
    {"role": "UX / UI", "fte": 0.3, "low": 160_000, "base": 210_000, "high": 280_000},
]


## Data

Каждая строка роли содержит средний FTE и три плановых оклада. Диапазон не является обещанием найма по указанной цене: он нужен для чувствительности бюджета и должен быть заменён внутренними ставками HR/Finance перед утверждением сметы.


In [2]:
total_fte = sum(row["fte"] for row in roles)
person_months = total_fte * months
role_rows = []
for row in roles:
    calculated = dict(row)
    for scenario in ("low", "base", "high"):
        calculated[f"loaded_{scenario}"] = row["fte"] * row[scenario] * months * loaded_multiplier
    role_rows.append(calculated)

totals = {
    scenario: sum(row[f"loaded_{scenario}"] for row in role_rows)
    for scenario in ("low", "base", "high")
}
budget_with_contingency = totals["base"] * (1 + contingency)

print(f"FTE: {total_fte:.1f}")
print(f"Человеко-месяцы: {person_months:.1f}")
for scenario in ("low", "base", "high"):
    print(f"{scenario.upper()}: {totals[scenario]:,.0f} ₽")
print(f"BASE + 15%: {budget_with_contingency:,.0f} ₽")


FTE: 7.3
Человеко-месяцы: 36.5
LOW: 11,926,200 ₽
BASE: 14,750,450 ₽
HIGH: 18,153,850 ₽
BASE + 15%: 16,963,018 ₽


In [3]:
assert abs(total_fte - 7.3) < 1e-9
assert abs(person_months - 36.5) < 1e-9
assert round(totals["low"]) == 11_926_200
assert round(totals["base"]) == 14_750_450
assert round(totals["high"]) == 18_153_850
assert round(budget_with_contingency) == 16_963_018
assert abs(sum(row["loaded_base"] for row in role_rows) - totals["base"]) < 0.01
print("Все контрольные суммы совпали.")


Все контрольные суммы совпали.


## Results

Ниже отдельно считаются сопровождение, инфраструктура, создание нового сценария и опциональный ML/LLM. Они не добавляются к ФОТ разработки автоматически, чтобы избежать двойного счёта.


In [4]:
support = [
    ("Backend", 0.40, 278_000), ("Frontend", 0.15, 260_000),
    ("QA", 0.20, 230_000), ("DevOps", 0.25, 280_000),
    ("Технолог", 0.25, 250_000), ("Методист", 0.15, 180_000),
    ("AppSec", 0.10, 250_000), ("PM", 0.10, 260_000),
]
support_fte = sum(row[1] for row in support)
annual_support = sum(row[1] * row[2] for row in support) * 12 * loaded_multiplier

pilot_infra_monthly = 3 * 1_800 + 3 * 180 + 3 * 80 * 6 + 235
pilot_infra_annual = pilot_infra_monthly * 12

scenario_work = [
    ("Технолог", 0.50, 250_000), ("Методист", 0.40, 180_000),
    ("Backend", 0.40, 278_000), ("Frontend", 0.20, 260_000),
    ("QA", 0.40, 230_000), ("PM", 0.15, 260_000),
]
scenario_person_months = sum(row[1] for row in scenario_work)
scenario_loaded_cost = sum(row[1] * row[2] for row in scenario_work) * loaded_multiplier

ml_llm_loaded_cost = 0.8 * 350_000 * months * loaded_multiplier
gpu_l4_hourly = 69.52
gpu_burst_monthly = gpu_l4_hourly * 100
gpu_always_on_monthly = gpu_l4_hourly * 730

print(f"Поддержка: {support_fte:.2f} FTE; {annual_support:,.0f} ₽/год")
print(f"Инфраструктура пилота с мониторингом: {pilot_infra_monthly:,.0f} ₽/мес; {pilot_infra_annual:,.0f} ₽/год")
print(f"Новый сценарий: {scenario_person_months:.2f} чел.-мес.; {scenario_loaded_cost:,.0f} ₽")
print(f"ML/LLM разработка: {ml_llm_loaded_cost:,.0f} ₽")
print(f"GPU L4: {gpu_burst_monthly:,.0f} ₽ за 100 ч/мес.; {gpu_always_on_monthly:,.0f} ₽ за 730 ч/мес.")


Поддержка: 1.60 FTE; 6,978,972 ₽/год
Инфраструктура пилота с мониторингом: 7,615 ₽/мес; 91,380 ₽/год
Новый сценарий: 2.05 чел.-мес.; 702,416 ₽
ML/LLM разработка: 2,002,000 ₽
GPU L4: 6,952 ₽ за 100 ч/мес.; 50,750 ₽ за 730 ч/мес.


In [5]:
github_team_10 = 4 * 10 * planning_fx_rub_per_usd
youtrack_10 = 0
youtrack_11 = 4.5 * 11 * planning_fx_rub_per_usd
grafana_cloud_pro = 19 * planning_fx_rub_per_usd
print(f"GitHub Team, 10 пользователей: {github_team_10:,.0f} ₽/мес.")
print(f"YouTrack, 10 пользователей: {youtrack_10:,.0f} ₽/мес.")
print(f"YouTrack, 11 пользователей: {youtrack_11:,.0f} ₽/мес.")
print(f"Grafana Cloud Pro: от {grafana_cloud_pro:,.0f} ₽/мес. плюс потребление")


GitHub Team, 10 пользователей: 3,400 ₽/мес.
YouTrack, 10 пользователей: 0 ₽/мес.
YouTrack, 11 пользователей: 4,208 ₽/мес.
Grafana Cloud Pro: от 1,615 ₽/мес. плюс потребление


In [6]:
assert abs(support_fte - 1.6) < 1e-9
assert round(annual_support) == 6_978_972
assert pilot_infra_monthly == 7_615
assert round(scenario_loaded_cost) == 702_416
assert round(ml_llm_loaded_cost) == 2_002_000
assert round(gpu_always_on_monthly) == 50_750
print("Итоговые проверки OPEX и опций пройдены.")


Итоговые проверки OPEX и опций пройдены.


## Takeaways

1. Главная статья пилота — люди и предметная приёмка, а не лицензии или VM.
2. Для бюджетной заявки разумный базовый ориентир ФОТ — 16,96 млн ₽ с резервом 15%; внешние услуги и покупка on-premise-оборудования идут отдельными строками.
3. После запуска базовое сопровождение требует около 1,6 FTE и 6,98 млн ₽ в год по тем же ставкам.
4. Опциональные ML/LLM добавляют около 2,0 млн ₽ проектного ФОТ; GPU следует арендовать по потреблению, если измеренная задержка CPU действительно неприемлема.
5. Перед утверждением Finance/HR должны заменить плановые оклады, режим страховых взносов и накладные на внутренние значения.
